In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import minmax_scale
import numpy as np
import ast

import zipfile
import os

import warnings
warnings.filterwarnings("ignore")

# Data retreiving

In [2]:
# Caminho do arquivo ZIP
zip_file_path = "./data/TAD_INSPECAO_measurement_202410181443_13.zip"

# Lista para armazenar todos os dataframes
dataframes = []

# Abrindo o arquivo ZIP
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # Para cada arquivo CSV dentro do ZIP
    for file_name in zip_ref.namelist():
        # Verificando se o arquivo é um CSV
        if file_name.endswith('.csv'):
            # Tentando ler o arquivo CSV
            with zip_ref.open(file_name) as file:
                try:
                    # Tentando com ponto e vírgula como delimitador
                    df = pd.read_csv(file, sep=';', on_bad_lines='skip', engine='python')
                    dataframes.append(df)
                    print(f"{file_name} lido com sucesso.")
                except pd.errors.ParserError:
                    print(f"Erro persistente no arquivo: {file_name}. Arquivo ignorado.")

# Agora a lista 'dataframes' contém todos os dataframes dos arquivos CSV que foram lidos corretamente

# Caso deseje unir todos os dataframes em um só
df_final = pd.concat(dataframes, ignore_index=True)

TAD_INSPECAO_measurement_202410181443_13.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_2.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_3.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_4.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_5.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_6.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_7.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_8.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_9.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_10.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_11.csv lido com sucesso.
TAD_INSPECAO_measurement_202410181443_12.csv lido com sucesso.


In [9]:
df_final['texto'] = df_final['texto'].apply(lambda x: x.strip())
df_final['texto'] = df_final['texto'].apply(lambda x: x.replace(" ","").replace("{\n","").replace("}","").replace("\n","").replace("'",""))

In [ ]:
df_final['teste'].str.split(",",expand=True).head(1)

In [14]:
df_final['texto'].head(10)

0    corrente:0,0,-0.06614785641431808,0.0661478564...
1    corrente:0,0,0,0.06614785641431808,18.19066047...
2    corrente:-0.06614785641431808,0,1.852140069007...
3    corrente:0,-0.06614785641431808,-0.06614785641...
4    corrente:0,0,0.06614785641431808,69.9182815551...
5    corrente:0.06614785641431808,0.066147856414318...
6    corrente:0.06614785641431808,0,0.0661478564143...
7    corrente:0,0,-0.06614785641431808,0,37.9027214...
8    corrente:0,0.06614785641431808,0.0661478564143...
9    corrente:0,0,0.26459142565727234,0.26459142565...
Name: texto, dtype: object

In [26]:
for x,y in df_final['teste'].str.split(",",expand=True).head(1).items():
    if 'corrente:' in y[0]:
        id1 = x
    if 'tensao:' in y[0]:
        id2 = x
    if 'pressao:' in y[0]:
        id3 = x

df_vc = df_final['teste'].str.split(",",expand=True).loc[:,id1:id3-1].copy()

In [52]:
df_vc = df_vc.apply(lambda x: x.replace(' ','').replace('corrente:','',regex=True).replace('tensao:','',regex=True),axis=1)

In [74]:
# Gera nomes para as 310 primeiras colunas (corrente)
nomes_corrente = [f"{i}_corrente" for i in range(310)]

# Gera nomes para as 310 últimas colunas (tensão)
nomes_tensao = [f"{i}_tensao" for i in range(310)]

# Junta os dois conjuntos de nomes
novos_nomes = nomes_corrente + nomes_tensao

# Aplica os novos nomes ao dataframe
df_vc.columns = novos_nomes


Index(['0_corrente', '1_corrente', '2_corrente', '3_corrente', '4_corrente',
       '5_corrente', '6_corrente', '7_corrente', '8_corrente', '9_corrente',
       ...
       '300_tensao', '301_tensao', '302_tensao', '303_tensao', '304_tensao',
       '305_tensao', '306_tensao', '307_tensao', '308_tensao', '309_tensao'],
      dtype='object', length=620)


In [77]:
df_full = df_final[ ['dataHora','valorMed','maquina','parametro','item','idMedicao']].copy()
df_full = pd.concat([df_full,df_vc],axis=1)

In [79]:
df_full.to_csv('./data/new_machine_2.csv')